# Julia notebook to compute the buoyancy field using the 1D vertical diffusion equation, inspired by Peterson & Callies (2025, in review) theory. Then compute the associated geostrophic flow using the frictional thermal wind equation.

twnh Sep '25

This notebook solves the steady vertical diffusion problem with a Green's function.

The problem is to solve:
\begin{align}
\epsilon^2 \kappa \frac{d^2 b}{dz^2} 
    + \gamma \left( B(z) - b \right) & = 0 , 
\end{align}
for $b(z)$, given buoyancy profile $B(z)$, relaxation coefficient $\gamma$, with boundary conditions
\begin{align}
 \frac{db}{dz}  &= 0 \text{~ at~} z = -H , \\
 b &= 0 \text{~ at~} z = 0 ,
\end{align}
where bottom buoyancy gradient is be zero.

This problem is solved repeatedly for different $H(x)$ fields and $B(x,z)$ fields. This constructs a buoyancy field that varies in $(x,z)$ using the 1D diffusion equation.

Then, given $b(x,z)$, solve:

\begin{align}
- f  v & = - \frac{\partial p}{\partial x} + \epsilon^2   \nu \frac{d^2 u}{d z^2}, \\
  f  u & = - \frac{\partial p}{\partial y} + \epsilon^2   \nu \frac{d^2 v}{d z^2} , 
\end{align}
for $u(z), v(z)$, given viscosity $\nu$, and boundary conditions
\begin{align}
\text{Surface~}z = 0:
\begin{cases}
\displaystyle \epsilon^2 \nu \partial_z u & = \tau^x \\
\displaystyle  \epsilon^2 \nu \partial_z v & = \tau^y 
\end{cases}
 \\
\text{Bottom~}z = -H:
\begin{cases}
u & = 0  \\
v & = 0 
\end{cases} ,
\end{align}
where $(\tau^x, \tau^y)$ is the known wind stress.

In [1]:
using SymPy
im = SymPy.im  # SymPy's imaginary unit

using Infiltrator

Define symbols and functions

In [2]:
# Geometry symbolic parameters:
z, ξ   = symbols("z ξ",   real=true, negative=true) # Vertical coordinate and source location (both in [-H,0])
x, y   = symbols("x y",   real=true)                # Horizontal coordinates
H      = SymFunction("H", real=true, positive=true) # Domain depth H(x)
α      = 1//2                                       # Aspect ratio value. Note // which maintains rational type
# Hfn    = α * (1 - x^2)                              # Bathymetry function
Hfn    = α * x                                      # Bathymetry function
Hfn    = 1                                          # Bathymetry function
geometry_params = (H(x,y), z, ξ)

# Frictional thermal wind equation symbolic parameters:
f, ϵ   = symbols("f ϵ",   real=true, positive=true) # Coriolis parameter and Ekman number
ν₀, ϕ  = symbols("ν₀ ϕ",  real=true, positive=true) # Viscosity parameters
τˣ, τʸ = symbols("τˣ τʸ", real=true)                # Surface wind stress components
τs     = τˣ + im * τʸ                               # Complex surface wind stress

# Define viscosity profile here:
ν = ν₀                                              # Constant viscosity profile

uv_params = (f, ϵ, ν)

# Buoyancy equation symbolic parameters:
κ₀, ψ  = symbols("κ₀ ψ",  real=true, positive=true) # Diffusivity parameters
γ      = symbols("γ",     real=true, positive=true) # Relaxation parameter

# Define diffusivity profile here:
κ = κ₀                                              # Constant viscosity profile

b_params = (κ, ϵ, γ) ;

### Set the problem parameters here:

In [3]:
# Define the $B(z)$ source term here:
B      = SymFunction("B")                           # Source term function B(z): buoyancy field relaxation profile.
Bfn    = z                                          # Linear profile with vanishing surface buoyancy

f_val  = 1
ϵ_val  = 1

γ_val  = 1

κ₀_val = 1

ν₀_val = κ₀_val

τˣ_val = 0
τʸ_val = 1

# Compute compound parameters:
# ϕ_val = sqrt(f_val / ν₀_val) / ϵ_val
# ψ_val = sqrt(γ_val / κ₀_val) / ϵ_val
ϕ_val = 1
ψ_val = 1
param_values = Dict(f=>f_val, ϵ=>ϵ_val, γ=>γ_val, κ₀=>κ₀_val, ν₀=>ν₀_val, ϕ=>ϕ_val, ψ=>ψ_val)
force_values = Dict(τˣ=>τˣ_val, τʸ=>τʸ_val)

Dict{Sym{PyCall.PyObject}, Int64} with 2 entries:
  τʸ => 1
  τˣ => 0

Code to solve the buoyancy equation using a Green's function:

In [4]:
function compute_Gb(b_params, geometry_params, param_values)
    # Setup symbols and parameters:
    κ, ϵ, γ = b_params
    H, z, ξ = geometry_params
    b       = SymFunction("b")
    A       = symbols("A",     real=true)                      # Unknown coefficient in the Green's function solution
    
    #0. Define the ODE for b(z):
    ode = Eq(ϵ^2 * κ * diff(diff(b(z),z),z) - diff(κ,z) * diff(b(z),z) - γ * b(z), 0)

    # 1. Solve for $G_- (z)$ on $-H \le z \le \xi$:
    bm = dsolve(ode, b(z), ics = Dict(diff(b(z),z).subs(z,-H)=>0)).rhs
    @assert simplify(diff(bm,z).subs(z,-H) - 0) == 0              # Check Neumann BC at bottom
    bm_const = filter(x -> startswith(string(x), "C"), bm.free_symbols)
    
    bm = bm.subs(first(bm_const), A)                              # Replace constant with A so it doesn't conflict later

    # 2. Solve for $G_+(z)$ on  $\xi \le z \le 0$:
    bp = dsolve(ode, b(z), ics = Dict(b(0)=>0) ).rhs
    @assert simplify(bp.subs(z,0)) == 0                           # Check Dirichlet BC at top

    # 3. Compute Wronskian $W(z)$:
    W = simplify(bm * diff(bp, z) - bp * diff(bm, z))

    # 4. Compute Green's function $G(z; \xi)$:
    Gm = simplify(bm * bp.subs(z,ξ) / (κ.subs(z,ξ) * W.subs(z,ξ)))
    Gp = simplify(bm.subs(z,ξ) * bp / (κ.subs(z,ξ) * W.subs(z,ξ)))

    # Check continuity and jump condition:
    @assert Gm.subs(z,ξ) - Gp.subs(z,ξ) == 0
    @assert simplify(diff(Gm, z).subs(z,ξ) - diff(Gp, z).subs(z,ξ)) + 1/κ.subs(z,ξ) == 0

    # Define piecewise Green's function:
    G = simplify(sympy.Piecewise((Gm, Le(z,ξ)), (Gp, Ge(z,ξ))))

    # Check boundary conditions are satisfied
    @assert simplify(diff(G,z).subs(z,-H).subs(ξ,-H//3).subs(H,1//2)) == 0
    @assert simplify(G.subs(z,0).subs(ξ,-H//3)) == 0

    return G
end

compute_Gb (generic function with 1 method)

Code to solve the frictional geostrophic equation using a Green's function:

In [5]:
function compute_Guv(uv_params, geometry_params, param_values)
    # Setup symbols and parameters:
    f, ϵ, ν = uv_params
    H, z, ξ = geometry_params
    uv      = SymFunction("uv")
    A       = symbols("A",     real=true)                      # Unknown coefficient in the Green's function solution

    #0. Define the ODE for d/dz(uv(z)) = uv(z):
    ode = Eq(im * f * uv(z) + ϵ^2 * diff(diff(ν * uv(z),z),z), 0)
    
    # 1. Solve for $G_- (z)$ on $-H \le z \le \xi$:
    Gₘ = dsolve(ode, uv(z), ics = Dict(uv(z).subs(z,-H(x,y))=>0)).rhs
    @assert simplify(ode.lhs.subs(uv(z),Gₘ)) == 0                               # Check solution
    # Replace constant names because otherwise they can interfere with the constants from the next dsolve below.
    const_names = collect([string(s) for s in Gₘ.free_symbols if occursin(r"^C\d+", string(s))])
    Gₘ = Gₘ.subs(const_names[1],A)
    @assert simplify(Gₘ.subs(z,-H(x,y))) == 0                                   # Check bottom BC

    # 2. Solve for $G_+(z)$ on  $\xi \le z \le 0$:
    Gₚ = dsolve(ode, uv(z), ics = Dict(diff(uv(z),z).subs(z,0)=>0)).rhs         # This is where surface forcing would go.
    @assert simplify(ode.lhs.subs(uv(z),Gₚ)) == 0                               # Check solution
    @assert simplify(diff(Gₚ,z).subs(z,0)) == 0                                 # Check surface BC

    # 3. Compute Wronskian $W(z)$:
    W = simplify(Gₘ * diff(Gₚ, z) - Gₚ * diff(Gₘ, z))

    # 4. Compute Green's function $G(z; \xi)$:
    Gm = simplify(Gₘ * Gₚ.subs(z,ξ) / (ν.subs(z,ξ) * W.subs(z,ξ)))
    Gp = simplify(Gₘ.subs(z,ξ) * Gₚ / (ν.subs(z,ξ) * W.subs(z,ξ)))

    # Check continuity and jump condition:
    @assert Gm.subs(z,ξ) - Gp.subs(z,ξ) == 0
    @assert simplify(diff(Gm, z).subs(z,ξ) - diff(Gp, z).subs(z,ξ)) + 1/ν.subs(z,ξ) == 0

    # #5. Define piecewise Green's function:
    G = simplify(sympy.Piecewise((Gm, Le(z,ξ)), (Gp, Ge(z,ξ))))
    
    # Check boundary conditions:
    @assert simplify(diff(G,z).subs(z,0).subs(ξ,-H//2)) == 0
    @assert simplify(G.subs(z,-H(x,y)).subs(ξ,-H//2)) == 0

    return G
end

compute_Guv (generic function with 1 method)

#### Define helper functions for substitutions

In [6]:
# Be careful with this function!!!
# It's more powerful than the .subs() method, but is more risky.
function tom_subs(expr,param_values)
    new_expr_str = string(expr)
    for (k, v) in param_values
        new_expr_str = replace(new_expr_str, string(k) => string(v))
    end
    return sympify(new_expr_str)
end

function get_var_in_expr(var,expr)
    filter(s -> string(s) == var, collect(expr.free_symbols))[1]
end

get_var_in_expr (generic function with 1 method)

# Compute the G's functions:

In [ ]:
Gb_sym  = compute_Gb(b_params,geometry_params,param_values) 
display("Full Gb(x,ξ):")
display(Gb_sym)
Guv_sym = compute_Guv(uv_params,geometry_params,param_values) 
display("Full Guv(x,ξ):")
display(Guv_sym)

"Full Gb(x,ξ):"

/  /          ___   \ /     ___                   ___ \    ___                 >
|  |     -2*\/ γ *ξ | | 2*\/ γ *H(x, y)    -2*z*\/ γ  |  \/ γ *(z + ξ)         >
|  |     -----------| | ---------------    -----------|  -------------         >
|  |        ____    | |      ____             ____    |      ____              >
|  |      \/ κ₀ *ϵ  | |    \/ κ₀ *ϵ         \/ κ₀ *ϵ  |    \/ κ₀ *ϵ            >
|ϵ*\1 - e           /*\e                + e           /*e                      >
|---------------------------------------------------------------------  for z  >
|                               /     ___            \                         >
|                               | 2*\/ γ *H(x, y)    |                         >
|                               | ---------------    |                         >
|                               |      ____          |                         >
|                    ___   ____ |    \/ κ₀ *ϵ        |                         >
|                2*\/ γ *\/ 

"Full Guv(x,ξ):"

/  /         ___   ____              \ /     ___     ____    \    ___     5/2  >
|  |     2*\/ f *\/ -I *(z + H(x, y))| | 2*\/ f *ξ*\/ -I     |  \/ f *(-I)   * >
|  |     ----------------------------| | ----------------    |  -------------- >
|  |                 ____            | |       ____          |          ____   >
|  |               \/ ν₀ *ϵ          | |     \/ ν₀ *ϵ        |        \/ ν₀ *ϵ >
|ϵ*\1 - e                            /*\e                 + 1/*e               >
|----------------------------------------------------------------------------- >
|                                      /     ___   ____            \           >
|                                      | 2*\/ f *\/ -I *H(x, y)    |           >
|                                      | ----------------------    |           >
|                                      |          ____             |           >
|                    ___   ____   ____ |        \/ ν₀ *ϵ           |           >
|                2*\/ f *\/ 

### For LaTeX write up:

$G_b$ with compound constant $\psi$

In [8]:
tmp = tom_subs(Gb_sym,Dict("γ"=>"ψ^2 * ϵ^2 * κ₀"))               # THIS IS FRAGILE!!!
tmp = tom_subs(tmp,Dict("sqrt(κ₀*ψ^2*ϵ^2)/(sqrt(κ₀)*ϵ)"=>"ψ"))
tmp = tom_subs(tmp,Dict("sqrt(-I)"=>"exp(-I*pi/4)"))
Gb  = tom_subs(tmp,Dict("sqrt(κ₀*ψ^2*ϵ^2)"=>"ψ*sqrt(κ₀)*ϵ"))
display("Simplified Gb(z,ξ)")
display(Gb)

tmp4 = tom_subs(Gb,Dict("ξ"=>"-H(x,y)"))
display("Simplified Gb(z,-H(x,y))")
display(simplify(tmp4.args[1].args[1]))

display("Simplified d/dx Gb(z,ξ)")
x_var = get_var_in_expr("x" ,Gb)
tmp5  = simplify(diff(Gb,x_var))
display(tmp5.args[1].args[1])

"Simplified Gb(z,ξ)"

//     -2*ξ*ψ\ / 2*ψ*H(x, y)    -2*z*ψ\  ψ*(z + ξ)            
|\1 - e      /*\e            + e      /*e                     
|-------------------------------------------------  for z <= ξ
|                   / 2*ψ*H(x, y)    \                        
|            2*κ₀*ψ*\e            + 1/                        
<                                                             
|/     -2*z*ψ\ / 2*ψ*H(x, y)    -2*ξ*ψ\  ψ*(z + ξ)            
|\1 - e      /*\e            + e      /*e                     
|-------------------------------------------------  for z >= ξ
|                   / 2*ψ*H(x, y)    \                        
\            2*κ₀*ψ*\e            + 1/                        

"Simplified Gb(z,-H(x,y))"

-cosh(ψ*(z + H(x, y)))*tanh(ψ*H(x, y)) 
---------------------------------------
                 κ₀*ψ                  

"Simplified d/dx Gb(z,ξ)"

/   2*z*ψ    2*ξ*ψ    2*ψ*(z + ξ)    \  ψ*(-z - ξ + 2*H(x, y)) d          
\- e      - e      + e            + 1/*e                      *--(H(x, y))
                                                               dx         
--------------------------------------------------------------------------
                     / 4*ψ*H(x, y)      2*ψ*H(x, y)    \                  
                  κ₀*\e            + 2*e            + 1/                  

Buoyancy profile $b(z)$ using the buoyancy Green's function and compound constant $\psi$

In [9]:
b_sym = simplify(integrate(-Gb_sym * B(ξ), (ξ,-H(x,y),0) ))            # Notice the horizontal dependence in H.
b_sym = tom_subs(b_sym, Dict("Max(z, -H(x, y))"=>"z"))
b_sym = tom_subs(b_sym, Dict("γ"=>"ψ^2 * ϵ^2 * κ₀"))
b_sym = tom_subs(b_sym, Dict("sqrt(κ₀*ψ^2*ϵ^2)/(sqrt(κ₀)*ϵ)"=>"ψ"))
b_sym = tom_subs(b_sym, Dict("sqrt(κ₀*ψ^2*ϵ^2)*H(x, y)/(sqrt(κ₀)*ϵ)"=>"ψ*H(x,y)"))
b_sym = tom_subs(b_sym, Dict("sqrt(κ₀*ψ^2*ϵ^2)*(z + H(x, y))/(sqrt(κ₀)*ϵ)"=>"ψ*(z+H(x,y))"))
b_sym = tom_subs(b_sym, Dict("sqrt(κ₀)*sqrt(κ₀*ψ^2*ϵ^2)"=>"κ₀*ψ*ϵ"))
b_sym = simplify(b_sym)
display("Simplified b(z,ξ):")
display(b_sym)

"Simplified b(z,ξ):"

/           /               z                  z              \                >
|           |               /                  /              |                >
|           |              |                  |               |                >
|/     z*ψ\ | 2*ψ*H(x, y)  |        ξ*ψ       |        -ξ*ψ   | / z*ψ    \   / >
|\1 - e   /*|e           * |  B(ξ)*e    dξ +  |  B(ξ)*e     dξ|*\e    + 1/ - \ >
|           |              |                  |               |                >
|           |             /                  /                |                >
\           \                                                 /                >
------------------------------------------------------------------------------ >
                                                                               >
                                                                               >

>           /              -H(x, y)                -H(x, y)             \      >
>           |              

Compute baroclinic potential energy $\chi$

In [10]:
x_var = get_var_in_expr("x",b_sym)
y_var = get_var_in_expr("y",b_sym)
z_var = get_var_in_expr("z",b_sym)
χ_sym = - simplify(integrate(expand(b_sym * z_var), (z_var, -H(x_var,y_var), 0) ))
χ_sym = tom_subs(χ_sym, Dict("(ψ > -oo) & (ψ < oo) & Ne(ψ, 0)"=>"true"))
display("Simplified χ(x,y):")
display(χ_sym)
χ = SymFunction("χ")

"Simplified χ(x,y):"

                                                                               >
                                                                               >
                                                                               >
                                       0                                       >
                                       /                                       >
  /                  ψ*H(x, y)     \  |                  /                  ψ* >
  |(-ψ*H(x, y) + 1)*e            1 |  |        -ξ*ψ      |(-ψ*H(x, y) + 1)*e   >
- |--------------------------- - --|* |  B(ξ)*e     dξ + |-------------------- >
  |             2                 2|  |                  |             2       >
  \            ψ                 ψ / /                   \            ψ        >
                                                                               >
                                                                               >
                            

χ

Define pressure in terms of a known part due to the buoyancy field $p_i$ and an unknown part due to the surface pressure $p_s$.

In [11]:
p  = SymFunction("p")                               # Total pressure function 
pₛ = SymFunction("pₛ")                               # Surface pressure function 
pᵢ = SymFunction("pᵢ")                              # Internal pressure function 
zp = symbols("zp", real=true, negative=true)
pᵢ_sym = integrate(expand(b_sym), (z, zp, 0))               # Compute internal pressure contribution from buoyancy
pᵢ_sym = tom_subs(pᵢ_sym,Dict(zp=>"z"))
pᵢ_sym = tom_subs(pᵢ_sym,Dict("Max(z, -H(x, y))"=>"z"))
pᵢ_sym = simplify(pᵢ_sym)
display("Internal pressure field pᵢ(x,y,z):")
display(pᵢ_sym)

x_var = get_var_in_expr("x",pᵢ_sym)
y_var = get_var_in_expr("y",pᵢ_sym)
z_var = get_var_in_expr("z",pᵢ_sym)
display("Total pressure field p(x,y,z):")
display(simplify(p))

pb = tom_subs(pᵢ_sym,Dict(z_var=>"-H(x,y)")) + pₛ(x_var,y_var)
display("Bottom pressure field p(x,y,z):")
display(simplify(pb))

# Specific example of the pressure field. It simplifies quite a bit.
expr  = simplify(tom_subs(pᵢ_sym,param_values))
expr2 = simplify(tom_subs(expr,Dict("B(ξ)"=>"ξ")))
expr2 = simplify(tom_subs(expr2,Dict("Ne(ψ^2, 0)"=>"true")))
display("Specific internal pressure field pᵢ(x,y,z):")
display(factor(expr2))

"Internal pressure field pᵢ(x,y,z):"

  /         z                         -H(x, y)                                 >
  |         /                             /                                    >
  |        |                             |                                     >
  | 2*z*ψ  |        -ξ*ψ       2*z*ψ     |          -ξ*ψ       2*ψ*(z + H(x, y >
z*|e     * |  B(ξ)*e     dξ - e     *    |    B(ξ)*e     dξ - e                >
  |        |                             |                                     >
  |       /                             /                                      >
  \                                                                            >
------------------------------------------------------------------------------ >
                                                                               >
                                                                               >

>      0                                      z                                >
>      /                   

"Total pressure field p(x,y,z):"

p

"Bottom pressure field p(x,y,z):"

                                   /  0                  -H(x, y)              >
                                   |  /                      /                 >
                                   | |                      |                  >
     / 2*ψ*H(x, y)    \            | |        -ξ*ψ          |          -ξ*ψ    >
κ₀*ψ*\e            + 1/*pₛ(x, y) + | |  B(ξ)*e     dξ -     |    B(ξ)*e     dξ >
                                   | |                      |                  >
                                   |/                      /                   >
                                   \                                           >
------------------------------------------------------------------------------ >
                                                                / 2*ψ*H(x, y)  >
                                                           κ₀*ψ*\e             >

>      0                 -H(x, y)            \                   
>      /                     /            

"Specific internal pressure field pᵢ(x,y,z):"

   /   z  2*H(x, y)      z    2*z  H(x, y)    H(x, y)\  -z 
-z*\z*e *e          + z*e  - e   *e        + e       /*e   
-----------------------------------------------------------
                       2*H(x, y)                           
                      e          + 1                       

$G_{uv}$ with compound constant $\phi$

In [12]:
tmp = tom_subs(Guv_sym,Dict("f"=>"ϕ^2 * ϵ^2 * ν₀"))               # THIS IS FRAGILE!!!
tmp = tom_subs(tmp,Dict("sqrt(ν₀*ϕ^2*ϵ^2)/(sqrt(ν₀)*ϵ)"=>"ϕ"))
# tmp = tom_subs(tmp,Dict("(-I)^(5/2)"=>"exp(-I*5*pi//4)","sqrt(-I)"=>"exp(-I*pi//4)"))         # sqrt(-I) = exp(-I*pi/4) etc. (principal value)
Guv = tom_subs(tmp,Dict("sqrt(ν₀*ϕ^2*ϵ^2)"=>"ϕ*sqrt(ν₀)*ϵ"))
display("Simplified Guv(z,ξ)")
display(Guv)

z_var = get_var_in_expr("z" ,Guv)
Guv0  = tom_subs(Guv,Dict("ξ"=>"0"))
Guv0  = simplify(Guv0.args[1].args[1])
display("Simplified Guv(z,0)")
display(Guv0)

tmp5  = diff(Guv, z_var)
tmp6 = tom_subs(tmp5,Dict(z_var=>"-H(x,y)"))
dGuv_dz_at_bottom = simplify(tmp6.args[1].args[1])
display("Simplified d/dz Guv(z,ξ) @ z = -H(x,y)")
display(dGuv_dz_at_bottom)

z_var = get_var_in_expr("z" ,Guv)
tmp7 = integrate(expand(Guv), (z_var, -H(x,y), 0))
expr = tmp7.args[1].args[1]
expr = tom_subs(expr,Dict("Max(ξ, -H(x, y))"=>"ξ"))
expr = tom_subs(expr,Dict("Min(0, ξ)"=>"ξ"))
Guv_int = simplify(factor(expr))
display("Simplified integral Guv(z,ξ) wrt z = -H(x,y) to 0")
display(Guv_int)

xi_var = get_var_in_expr("ξ" ,Guv_int)
display("Simplified integral Guv(z,ξ) wrt z = -H(x,y) to 0 @ ξ = 0")
Guv_int0 = tom_subs(expr,Dict(xi_var=>"0"))
display(simplify(factor(Guv_int0)))

"Simplified Guv(z,ξ)"

//           ____              \ /         ____    \        5/2                >
||     2*ϕ*\/ -I *(z + H(x, y))| | 2*ξ*ϕ*\/ -I     |  ϕ*(-I)   *(z + ξ)        >
|\1 - e                        /*\e             + 1/*e                         >
|----------------------------------------------------------------------  for z >
|                             /       ____            \                        >
|                        ____ | 2*ϕ*\/ -I *H(x, y)    |                        >
|               2*ν₀*ϕ*\/ -I *\e                   + 1/                        >
<                                                                              >
|/           ____              \ /         ____    \        5/2                >
||     2*ϕ*\/ -I *(ξ + H(x, y))| | 2*z*ϕ*\/ -I     |  ϕ*(-I)   *(z + ξ)        >
|\1 - e                        /*\e             + 1/*e                         >
|----------------------------------------------------------------------  for z >
|                           

"Simplified Guv(z,0)"

/           ____              \          5/2
|     2*ϕ*\/ -I *(z + H(x, y))|  z*ϕ*(-I)   
\1 - e                        /*e           
--------------------------------------------
               /       ____            \    
          ____ | 2*ϕ*\/ -I *H(x, y)    |    
   ν₀*ϕ*\/ -I *\e                   + 1/    

"Simplified d/dz Guv(z,ξ) @ z = -H(x,y)"

 /         ____    \        5/2               
 | 2*ξ*ϕ*\/ -I     |  ϕ*(-I)   *(ξ - H(x, y)) 
-\e             + 1/*e                        
----------------------------------------------
            /       ____            \         
            | 2*ϕ*\/ -I *H(x, y)    |         
         ν₀*\e                   + 1/         

"Simplified integral Guv(z,ξ) wrt z = -H(x,y) to 0"

  /       ____            5/2        ____                  5/2        \        >
  | ξ*ϕ*\/ -I     ξ*ϕ*(-I)       ϕ*\/ -I *H(x, y)    ϕ*(-I)   *H(x, y)|  ϕ*(-I >
I*\e           + e            - e                 - e                 /*e      >
------------------------------------------------------------------------------ >
                                   /       ____            \                   >
                                 2 | 2*ϕ*\/ -I *H(x, y)    |                   >
                             ν₀*ϕ *\e                   + 1/                   >

>  9/2        
> )   *H(x, y)
>             
> ------------
>             
>             
>             

"Simplified integral Guv(z,ξ) wrt z = -H(x,y) to 0 @ ξ = 0"

  /         ____        \ /           5/2        \        9/2        
  |     ϕ*\/ -I *H(x, y)| |     ϕ*(-I)   *H(x, y)|  ϕ*(-I)   *H(x, y)
I*\1 - e                /*\1 - e                 /*e                 
---------------------------------------------------------------------
                         /       ____            \                   
                       2 | 2*ϕ*\/ -I *H(x, y)    |                   
                   ν₀*ϕ *\e                   + 1/                   

Compute vertically-integrated velocity field

In [13]:
x_var  = get_var_in_expr("x" ,Guv_int)
y_var  = get_var_in_expr("y" ,Guv_int)
integrand = Guv_int * (diff(p(x_var,y_var,xi_var),x_var) + im * diff(p(x_var,y_var,xi_var),y_var))
xi_var = get_var_in_expr("ξ" ,integrand)
𝔘1 = integrate(expand(integrand),(xi_var,-H(x_var,y_var),0))
display("Buoyancy-driven flow field 𝔘₁(x,y,z):")
display(𝔘1)

nu_var = get_var_in_expr("ν₀" ,Guv_int0)
𝔘2 = simplify(Guv_int0 * τs / (nu_var*ϵ^2))
display("Stress-driven   flow field 𝔘₂(x,y,z):")
display(𝔘2)
𝔘 = simplify(𝔘1 + 𝔘2)

"Buoyancy-driven flow field 𝔘₁(x,y,z):"

  /     0                                        0                             >
  |     /                                        /                             >
  |    |                                        |                              >
  |    |           ____                         |             5/2              >
  |    |     ξ*ϕ*\/ -I  d                       |     ξ*ϕ*(-I)    d            >
I*|    |    e          *--(p(x, y, ξ)) dξ +     |    e           *--(p(x, y, ξ >
  |    |                dx                      |                 dx           >
  |    |                                        |                              >
  |   /                                        /                               >
  \-H(x, y)                                 -H(x, y)                           >
------------------------------------------------------------------------------ >
                                                                               >
                            

"Stress-driven   flow field 𝔘₂(x,y,z):"

              /  /       ____                    ____            \        5/2  >
              |  | 4*ϕ*\/ -I *H(x, y)      2*ϕ*\/ -I *H(x, y)    |  ϕ*(-I)   * >
I*(I*τʸ + τˣ)*\- \e                   + 2*e                   + 1/*e           >
------------------------------------------------------------------------------ >
                                               /       ____                    >
                                       2  2  2 | 4*ϕ*\/ -I *H(x, y)      2*ϕ*\ >
                                     ν₀ *ϕ *ϵ *\e                   + 2*e      >

>                    ____            \         5/2        
> H(x, y)      2*ϕ*\/ -I *H(x, y)    |  -ϕ*(-I)   *H(x, y)
>         + 2*e                   + 2/*e                  
> --------------------------------------------------------
>  ____            \                                      
> / -I *H(x, y)    |                                      
>               + 1/                                      

/                                                                              >
|                                 0                                            >
|                                 /                                            >
|               ____             |                                   ____      >
|        2  ϕ*\/ -I *H(x, y)     |    d                       2  ϕ*\/ -I *H(x, >
|- I*ν₀*ϵ *e                *    |    --(p(x, y, ξ)) dξ + ν₀*ϵ *e              >
|                                |    dx                                       >
|                                |                                             >
|                               /                                              >
\                            -H(x, y)                                          >
------------------------------------------------------------------------------ >
                                                                               >
                            

Compute bottom stress

In [14]:
x_var = get_var_in_expr("x",dGuv_dz_at_bottom)
y_var = get_var_in_expr("y",dGuv_dz_at_bottom)
xi_var = get_var_in_expr("ξ" ,integrand)
integrand = dGuv_dz_at_bottom * (diff(p(x_var,y_var,xi_var),x_var) + im * diff(p(x_var,y_var,xi_var),y_var))
τb1 = simplify(integrate(expand(integrand),(xi_var,-H(x_var,y_var),0))) 
display("Buoyancy-driven bottom stress ν d/dz u(x,y,z) @ z = -H:")
display(τb1)

nu_var = get_var_in_expr("ν₀" ,dGuv_dz_at_bottom)
dGuv_dz_at_bottom_and_top = tom_subs(dGuv_dz_at_bottom,Dict(xi_var=>"0"))  
τb2 = simplify(dGuv_dz_at_bottom_and_top * τs / (nu_var*ϵ^2))
display("Stress-driven   bottom stress ν d/dz u(x,y,z) @ z = -H")
display(τb2)
τb = simplify(τb1 + τb2)

"Buoyancy-driven bottom stress ν d/dz u(x,y,z) @ z = -H:"

/       0                                          0                           >
|       /                                          /                           >
|      |                                          |                            >
|      |           ____                           |           ____             >
|      |     ξ*ϕ*\/ -I  d                         |     ξ*ϕ*\/ -I  d           >
|-     |    e          *--(p(x, y, ξ)) dξ - I*    |    e          *--(p(x, y,  >
|      |                dx                        |                dy          >
|      |                                          |                            >
|     /                                          /                             >
\  -H(x, y)                                   -H(x, y)                         >
------------------------------------------------------------------------------ >
                                                                               >
                            

"Stress-driven   bottom stress ν d/dz u(x,y,z) @ z = -H"

                      9/2        
                ϕ*(-I)   *H(x, y)
2*(-I*τʸ - τˣ)*e                 
---------------------------------
       /       ____            \ 
  2  2 | 2*ϕ*\/ -I *H(x, y)    | 
ν₀ *ϵ *\e                   + 1/ 

/        /     0                                          0                    >
|        |     /                                          /                    >
|        |    |                                          |                     >
|        |    |           ____                           |           ____      >
|      2 |    |     ξ*ϕ*\/ -I  d                         |     ξ*ϕ*\/ -I  d    >
|- ν₀*ϵ *|    |    e          *--(p(x, y, ξ)) dξ + I*    |    e          *--(p >
|        |    |                dx                        |                dy   >
|        |    |                                          |                     >
|        |   /                                          /                      >
\        \-H(x, y)                                   -H(x, y)                  >
------------------------------------------------------------------------------ >
                                                                               >
                            

Work out some examples
(i) no buoyancy field $b=0$, flat bottom $H(x,y) = H$, specified surface pressure $p_s$

In [ ]:
xi_var = get_var_in_expr("ξ" ,Guv)
tmp = integrate(expand(Guv), (xi_var, -H(x,y), 0)).args[1].args[1]
tmp = tom_subs(tmp,Dict("Max(z, -H(x, y))"=>"z"))
tmp = tom_subs(tmp,Dict("Min(0, z)"=>"z"))
tmp = simplify(tmp * diff(pₛ(x),x))
display("Simplified integral Guv(z,ξ) * d/dx pₛ(x) wrt ξ = -H(x,y) to 0")
display(tmp)

### Form final equation for $\frak{U}$:

In [15]:
lhs = im * f * 𝔘
x_var = get_var_in_expr("x",pb)
y_var = get_var_in_expr("y",pb)
rhs = H(x_var,y_var)*(diff(pb,x_var) + im * diff(pb,y_var)) - (diff(χ(x_var,y_var),x_var) + im * diff(χ(x_var,y_var),y_var)) - ϵ^2 * τb + τs
final_eqn = Eq(lhs, rhs)
println()
display("Final equation for surface pressure pₛ(x,y):")
display(final_eqn)

"Final equation for surface pressure pₛ(x,y):"

    /                                                                          >
    |                                 0                                        >
    |                                 /                                        >
    |               ____             |                                   ____  >
    |        2  ϕ*\/ -I *H(x, y)     |    d                       2  ϕ*\/ -I * >
I*f*|- I*ν₀*ϵ *e                *    |    --(p(x, y, ξ)) dξ + ν₀*ϵ *e          >
    |                                |    dx                                   >
    |                                |                                         >
    |                               /                                          >
    \                            -H(x, y)                                      >
------------------------------------------------------------------------------ >
                                                                               >
                            

Work out specific simple example of this equation with no baroclinicity and a flat bottom

In [16]:
Hfn = 1
tmp  = tom_subs(final_eqn,Dict("p(x, y, ξ)"=>"pₛ(x)","B(ξ)"=>"0","B(-H(x, y))"=>"0","χ(x, y)"=>"0"))
tmp  = tom_subs(tmp,Dict("H(x, y)"=>Hfn,"pₛ(x, y)"=>"pₛ(x)"))
tmp  = tom_subs(tmp,param_values)
# x_var = get_var_in_expr("x",tmp)
# tmp  = tom_subs(tmp,Dict("τˣ"=>0,"τʸ"=>x_var))
tmp  = tom_subs(tmp,Dict("τˣ"=>0,"τʸ"=>"τʸ(x)"))
tmp  = tmp.doit(manual=true)
tmp = expand(tmp)
display("Final equation for pₛ(x) with no buoyancy and along-slope wind stress:")
display(tmp)

"Final equation for pₛ(x) with no buoyancy and along-slope wind stress:"

                                                                               >
                                                      /    5/2\                >
                                                      \(-I)   /                >
              2*I*τʸ(x)                      I*τʸ(x)*e                         >
- --------------------------------- + --------------------------------- + ---- >
       ____  /    5/2\    /    5/2\        ____  /    5/2\    /    5/2\        >
   2*\/ -I   \(-I)   /    \(-I)   /    2*\/ -I   \(-I)   /    \(-I)   /    2*\ >
  e        *e          + e            e        *e          + e            e    >

>                                                                              >
>                 ____                                d                        >
>               \/ -I                                 --(pₛ(x))                >
>      I*τʸ(x)*e                                      dx                       >
> -------------------------

In [17]:
eqn = expand(tmp.lhs - tmp.rhs)
tmp10 = tom_subs(eqn,Dict("Derivative(pₛ(x), x)"=>"xxx")).n()
xxx_var = get_var_in_expr("xxx",tmp10)
dspdx_coeffs = expand(tmp10).coeff(xxx_var)
rest = expand(tmp10 - dspdx_coeffs * xxx_var)
display("Coefficient of dpₛ/dx:")
display(dspdx_coeffs)
display("Rest of equation:")
display(rest)

x_var = get_var_in_expr("x",rest)
this_ode = Eq(dspdx_coeffs * diff(pₛ(x_var),x_var) + rest, 0)
display("Final ODE for pₛ(x):")
display(this_ode)   

ps_soln = dsolve(this_ode, pₛ(x))
display("General solution for pₛ(x):")
display(ps_soln)
display(ps_soln.n())

"Coefficient of dpₛ/dx:"

                                                 1                             >
1.11022302462516e-16 + ------------------------------------------------------  >
                                               2.5                        2.5  >
                       0.240142431175076*I*(-I)    + 1.03791252182696*(-I)     >

>                             1                                                >
> - ------------------------------------------------------ + ----------------- >
>                        0.5                           0.5                     >
>   1.91671626566601*(-I)    - 0.997222773345665*I*(-I)      - 0.9972227733456 >

>   0.541863457045632                                         1.31753840877988 >
> --------------------------------------- + ---------------------------------- >
>          2.5                        2.5                           0.5        >
> 65*I*(-I)    + 1.91671626566601*(-I)      - 4.0629286515035*I*(-I)    + 1.64 >

> *I                     

"Rest of equation:"

0.854471642673866*τʸ(x) - 1.64234084884427*I*τʸ(x)

"Final ODE for pₛ(x):"

                                                     /                         >
0.854471642673866*τʸ(x) - 1.64234084884427*I*τʸ(x) + |1.11022302462516e-16 + - >
                                                     |                         >
                                                     \                       0 >

>                          1                                                   >
> ----------------------------------------------------- - -------------------- >
>                        2.5                        2.5                        >
> .240142431175076*I*(-I)    + 1.03791252182696*(-I)      1.91671626566601*(-I >

>       1                                                 0.541863457045632    >
> ---------------------------------- + --------------------------------------- >
>  0.5                           0.5                             2.5           >
> )    - 0.997222773345665*I*(-I)      - 0.997222773345665*I*(-I)    + 1.91671 >

>                        

PyCall.PyError: PyError ($(Expr(:escape, :(ccall(#= /Users/twnh/.julia/packages/PyCall/1gn3u/src/pyfncall.jl:43 =# @pysym(:PyObject_Call), PyPtr, (PyPtr, PyPtr, PyPtr), o, pyargsptr, kw))))) <class 'ValueError'>
ValueError('0.854471642673866*τʸ(x) - 1.64234084884427*I*τʸ(x) + (1.11022302462516e-16 + 1/(0.240142431175076*I*(-I)**2.5 + 1.03791252182696*(-I)**2.5) - 1/(1.91671626566601*(-I)**0.5 - 0.997222773345665*I*(-I)**0.5) + 0.541863457045632/(-0.997222773345665*I*(-I)**2.5 + 1.91671626566601*(-I)**2.5) + 1.31753840877988*I/(-4.0629286515035*I*(-I)**0.5 + 1.64143546156249*(-I)**0.5) - 0.541863457045632/(-4.0629286515035*I*(-I)**0.5 + 1.64143546156249*(-I)**0.5) + 5.55111512312578e-17*I + 1.54186345704563/(-8.42713548247884*I*(-I)**0.5 - 2.82219519520608*(-I)**0.5) + 1.31753840877988*I/(1.64143546156249*(-I)**2.5 - 4.0629286515035*I*(-I)**2.5) - 1.31753840877988*I/(-8.42713548247884*I*(-I)**0.5 - 2.82219519520608*(-I)**0.5) - 1.31753840877988*I/(-0.997222773345665*I*(-I)**2.5 + 1.9167